In [6]:
import pandas as pd
import requests
import re
import time
from tqdm import tqdm
import os
import zipfile
from google.colab import files
from getpass import getpass

# =========================================================
# CONFIGURAÇÃO
# =========================================================
TOKEN = getpass("Digite seu GitHub Personal Access Token: ")
print("Token recebido:", len(TOKEN), "caracteres")


HEADERS={
    "Authorization":f"token {TOKEN}",
    "Accept":"application/vnd.github+json"
}

session=requests.Session()
session.headers.update(HEADERS)

BASE_DIR="dataset_commits_prs"
os.makedirs(BASE_DIR,exist_ok=True)


def clean(text):
    if not text:
        return ""
    return re.sub(r"\s+"," ",str(text)).strip()


def parse_commit_url(url):
    m=re.search(r"github\.com/([^/]+)/([^/]+)/commit/([a-f0-9]+)",str(url))
    if not m:
        return None,None,None
    return m.group(1),m.group(2),m.group(3)


def get(url):

    while True:

        r=session.get(url)

        if r.status_code==403:
            print("Rate limit...")
            time.sleep(60)
            continue

        if r.status_code in [404,422]:
            return None

        if r.status_code!=200:
            print(r.status_code,url)
            return None

        return r.json()


def get_prs(owner,repo,sha):

    prs=get(
        f"https://api.github.com/repos/{owner}/{repo}/commits/{sha}/pulls"
    )

    return prs or []


def unique_prs(prs):

    seen=set()
    result=[]

    for pr in prs:

        n=pr.get("number")

        if not n or n in seen:
            continue

        seen.add(n)
        result.append(pr)

    return result


def issue_comments(owner,repo,n):
    return get(f"https://api.github.com/repos/{owner}/{repo}/issues/{n}/comments") or []


def review_comments(owner,repo,n):
    return get(f"https://api.github.com/repos/{owner}/{repo}/pulls/{n}/comments") or []


def reviews(owner,repo,n):
    return get(f"https://api.github.com/repos/{owner}/{repo}/pulls/{n}/reviews") or []


def extract_comments(items):

    txt=[]

    for i in items:

        user=i.get("user",{})

        if user.get("type")=="Bot":
            continue

        body=clean(i.get("body"))

        if body=="":
            continue

        txt.append(
            f"{user.get('login','unknown')}\n{body}"
        )

    return "\n\n------------------------------\n\n".join(txt)


def extract_reviews(items):

    txt=[]

    for i in items:

        user=i.get("user",{})

        if user.get("type")=="Bot":
            continue

        body=clean(i.get("body"))

        if body=="":
            continue

        txt.append(
            f"{user.get('login','unknown')} ({i.get('state','')})\n{body}"
        )

    return "\n\n------------------------------\n\n".join(txt)


df=pd.read_csv("commits_refactorings_200.csv")

url_col=[c for c in df.columns if "url" in c.lower()][0]

rows=[]

KeyboardInterrupt: Interrupted by user

In [ ]:
# ==========================================================
# LOOP PRINCIPAL
# ==========================================================

for i, url in tqdm(list(enumerate(df[url_col])), total=len(df)):

    owner, repo, sha = parse_commit_url(url)

    if sha is None:
        continue

    project_dir = os.path.join(BASE_DIR, f"{owner}_{repo}")
    os.makedirs(project_dir, exist_ok=True)

    commit = get(
        f"https://api.github.com/repos/{owner}/{repo}/commits/{sha}"
    )

    if commit is None:
        continue

    commit_message = clean(commit["commit"]["message"])

    commit_url = f"https://github.com/{owner}/{repo}/commit/{sha}"

    prs = unique_prs(
        get_prs(owner, repo, sha)
    )

    pr_blocks = []
    pr_ids = []

    for pr_meta in prs:

        number = pr_meta["number"]

        if number in pr_ids:
            continue

        pr = get(
            f"https://api.github.com/repos/{owner}/{repo}/pulls/{number}"
        )

        if pr is None:
            continue

        title = clean(pr.get("title"))
        body = clean(pr.get("body"))

        issue_text = extract_comments(
            issue_comments(owner, repo, number)
        )

        review_comment_text = extract_comments(
            review_comments(owner, repo, number)
        )

        review_text = extract_reviews(
            reviews(owner, repo, number)
        )

        # ignora PR totalmente vazia
        if (
            title == ""
            and body == ""
            and issue_text == ""
            and review_comment_text == ""
            and review_text == ""
        ):
            continue

        pr_ids.append(number)

        pr_blocks.append(f"""
==================================================
PR #{number}

TITLE
{title}

BODY
{body}

ISSUE COMMENTS
{issue_text}

REVIEW COMMENTS
{review_comment_text}

REVIEWS
{review_text}
==================================================
""")

    case_id = f"C{i+1:03d}"

    text = f"""
CASE
{case_id}

PROJECT
{owner}/{repo}

COMMIT URL
{commit_url}

COMMIT SHA
{sha}

COMMIT MESSAGE
{commit_message}

PULL REQUESTS

{''.join(pr_blocks)}
"""

    txt_path = os.path.join(
        project_dir,
        f"{case_id}_{sha[:7]}.txt"
    )

    with open(
        txt_path,
        "w",
        encoding="utf-8"
    ) as f:
        f.write(text)

    rows.append({
        "case": case_id,
        "project": f"{owner}/{repo}",
        "commit_sha": sha,
        "commit_url": commit_url,
        "commit_message": commit_message,
        "pr_ids": ";".join(map(str, pr_ids)),
        "text": text
    })

    time.sleep(0.3)


# ==========================================================
# CSV PARA O ATLAS.ti
# ==========================================================

csv_path = os.path.join(BASE_DIR, "corpus.csv")

pd.DataFrame(rows).to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ==========================================================
# ZIP ÚNICO
# ==========================================================

zip_name = "dataset_commits_prs.zip"

with zipfile.ZipFile(
    zip_name,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, _, files_list in os.walk(BASE_DIR):

        for file in files_list:

            full = os.path.join(root, file)

            arc = os.path.relpath(
                full,
                BASE_DIR
            )

            zipf.write(full, arc)


print(f"Arquivos TXT: {len(rows)}")
print(f"CSV: {csv_path}")
print(f"ZIP: {zip_name}")

files.download(zip_name)

100%|██████████| 200/200 [07:52<00:00,  2.36s/it]

Arquivos TXT: 200
CSV: dataset_commits_prs/corpus.csv
ZIP: dataset_commits_prs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>